# Semana 3: ambiente dos três regimes de ajuste fino

Primeiro item da semana 3 do doc 20b (fase 5 do doc 20). Instala o TerraTorch na versão fixada, registra o backbone pré-treinado por revisão e hash, e constrói os três regimes com os pesos pré-treinados carregados, conferindo as contagens de parâmetros treináveis do doc 22, seção 11.1.

| Regime | O que treina | Esperado |
|---|---|---|
| R1, codificador congelado | pescoço, UperNet, cabeça | 15,1 M |
| R2, LoRA | LoRA (r = 16, α = 16 em q, v, fc1, fc2), pescoço, UperNet, cabeça | 20,6 M |
| R3, ajuste completo | tudo | 319,0 M |

**Nenhum treino roda aqui e nenhum dado é lido.** Precisa de GPU só para a conferência de passagem direta; serve a T4. Abra o Colab logada na conta do mestrado.

1. Instalação com versões exatas.
2. Drive e pastas (se o Drive não montar, segue com disco local).
3. Backbone pré-treinado: revisão do Hugging Face, hash e cópia no Drive.
4. Construção dos três regimes e conferências.
5. Registro em `resultados/semana3/ambiente_regimes.json`.

In [ ]:
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

In [ ]:
import os, json, hashlib, platform, datetime

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = '/content/drive/MyDrive/Mestrado'
    assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'
    DRIVE_OK = True
except Exception as erro:
    print(f'Drive não montado ({type(erro).__name__}: {erro}).')
    print('Seguindo com disco local: o backbone e o JSON ficam só nesta sessão. Cole a saída no Claude;')
    print('o arquivo definitivo no Drive fica para uma segunda execução, com o Drive montado.')
    RAIZ = '/content/Mestrado_local'
    os.makedirs(f'{RAIZ}/modelo_fixado', exist_ok=True)
    DRIVE_OK = False

BACKBONE_DIR = f'{RAIZ}/modelo_fixado/backbone'
RES = f'{RAIZ}/resultados/semana3'
os.makedirs(BACKBONE_DIR, exist_ok=True)
os.makedirs(RES, exist_ok=True)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

## Backbone pré-treinado

O checkpoint ajustado da IBM (R0) tem commit e hash fixados desde a semana 1. O ponto de partida dos três regimes novos é outro arquivo, o backbone `Prithvi_EO_V2_300M_TL.pt` do repositório `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL`, que o TerraTorch baixava sem revisão fixada (doc 25, seção 1.1). Esta célula:

- resolve o commit `63adbd3` (o mais recente na consulta de 10/10/2026) para o hash completo;
- lê o hash do arquivo em todos os commits em que ele existe, para saber se o arquivo mudou ao longo do histórico;
- baixa o arquivo na revisão fixada para o Drive e confere o sha256 contra o do Hugging Face.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO_CURTA = '63adbd3'

api = HfApi()
commits = api.list_repo_commits(REPO)
alvo = [c for c in commits if c.commit_id.startswith(REVISAO_CURTA)]
assert len(alvo) == 1, f'Commit {REVISAO_CURTA} não encontrado ou ambíguo.'
REVISAO = alvo[0].commit_id

historico = []
for c in reversed(commits):
    achado = api.get_paths_info(REPO, ARQUIVO, revision=c.commit_id)
    if achado:
        assert achado[0].lfs is not None, f'{ARQUIVO} não veio como arquivo LFS em {c.commit_id[:7]}.'
        historico.append((c.commit_id[:7], c.created_at.date().isoformat(), achado[0].lfs.sha256))
hashes_distintos = sorted({h for _, _, h in historico})
print(f'Commits em que o arquivo existe: {len(historico)} de {len(commits)}')
print(f'Primeiro: {historico[0][0]} ({historico[0][1]}). Último: {historico[-1][0]} ({historico[-1][1]})')
print(f'Hashes distintos do arquivo no histórico: {len(hashes_distintos)}')
for h in hashes_distintos:
    print('  ', h)

sha_hub = api.get_paths_info(REPO, ARQUIVO, revision=REVISAO)[0].lfs.sha256

BACKBONE_PT = hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=BACKBONE_DIR)
sha_local = sha256_arquivo(BACKBONE_PT)
assert sha_local == sha_hub, 'sha256 do arquivo baixado difere do publicado pelo Hugging Face.'
print(f'\nRevisão fixada: {REVISAO}')
print(f'sha256 conferido: {sha_local}')
print(f'Arquivo: {BACKBONE_PT}')

## Construção dos regimes

Receita da IBM (`config.yaml` do commit `918b9f1`): backbone `prithvi_eo_v2_300_tl`, necks nos índices 5, 11, 17 e 23, UperNet de 256 canais com `head_dropout` 0,1, duas classes, `rescale` ligado. O backbone é carregado do arquivo fixado (`backbone_ckpt_path`). As diferenças entre os regimes são só as abaixo.

- **R1:** `freeze_backbone: true`.
- **R2:** `peft_config` com LoRA em `qkv.q_linear`, `qkv.v_linear`, `mlp.fc1` e `mlp.fc2`, r = 16, α = 16, com a matriz `qkv` separada em q, k e v (`replace_qkv`), como na configuração `peft-geofm` da IBM. `freeze_backbone` fica falso: o TerraTorch recusa LoRA com o backbone congelado.
- **R3:** nada congelado.

Conferências por regime: contagem de parâmetros treináveis contra o doc 22, tensores do codificador idênticos ao arquivo fixado (R1 e R3), 96 módulos LoRA no R2 (24 blocos × 4 alvos) e uma passagem direta com saída de forma (2, 2, 224, 224).

In [ ]:
import gc, warnings
import torch
try:
    from terratorch.tasks import SemanticSegmentationTask
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 (instalação) antes e confira que '
                       'ela imprime terratorch 1.2.11. Se o ambiente foi recriado, rode também as células 2 e 3 de novo.') from erro

warnings.filterwarnings('ignore', category=UserWarning)

BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def montar(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True,
    )
    if regime == 'R2':
        args['peft_config'] = LORA
    tarefa = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=1e-3, optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        freeze_backbone=(regime == 'R1'),
    )
    tarefa.configure_models()
    return tarefa

def tensores_diferentes_do_arquivo(codificador, estado_arquivo):
    diferentes, sem_par, comparados = [], [], 0
    for nome, tensor in codificador.state_dict().items():
        if 'pos_embed' in nome:
            continue
        origem = estado_arquivo.get('encoder.' + nome)
        if origem is None:
            sem_par.append(nome)
        elif torch.equal(tensor.cpu(), origem):
            comparados += 1
        else:
            diferentes.append(nome)
    return comparados, diferentes, sem_par

estado_arquivo = torch.load(BACKBONE_PT, map_location='cpu', weights_only=True)
REGISTRO = {}

for regime in ('R1', 'R2', 'R3'):
    tarefa = montar(regime)
    treinaveis = sum(p.numel() for p in tarefa.parameters() if p.requires_grad)
    total = sum(p.numel() for p in tarefa.parameters())
    no_codificador = sum(p.numel() for p in tarefa.model.encoder.parameters() if p.requires_grad)
    linha = dict(treinaveis=treinaveis, total=total, treinaveis_no_codificador=no_codificador)

    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, (
        f'{regime}: {treinaveis / 1e6:.2f} M treináveis, esperado {ESPERADO_M[regime]} M.')

    if regime in ('R1', 'R3'):
        comparados, diferentes, sem_par = tensores_diferentes_do_arquivo(tarefa.model.encoder, estado_arquivo)
        assert not diferentes, f'{regime}: tensores diferentes do arquivo fixado: {diferentes[:5]}'
        linha.update(tensores_conferidos=comparados, tensores_sem_par_no_arquivo=sem_par)
    else:
        n_lora = sum(1 for n, _ in tarefa.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, f'R2: {n_lora} módulos LoRA, esperado 96.'
        linha.update(modulos_lora=n_lora)

    tarefa.model.to(DISPOSITIVO).eval()
    with torch.no_grad():
        saida = tarefa.model(torch.randn(2, 6, 224, 224, device=DISPOSITIVO)).output
    assert tuple(saida.shape) == (2, 2, 224, 224), f'{regime}: saída {tuple(saida.shape)}'
    linha['forma_da_saida'] = list(saida.shape)

    REGISTRO[regime] = linha
    print(f"{regime}: treináveis {treinaveis / 1e6:.2f} M ({100 * treinaveis / total:.1f}% de {total / 1e6:.1f} M), "
          f"no codificador {no_codificador / 1e6:.2f} M, saída {tuple(saida.shape)}"
          + (f", tensores conferidos {linha['tensores_conferidos']}" if 'tensores_conferidos' in linha else '')
          + (f", módulos LoRA {linha['modulos_lora']}" if 'modulos_lora' in linha else ''))
    if linha.get('tensores_sem_par_no_arquivo'):
        print('   sem par no arquivo:', linha['tensores_sem_par_no_arquivo'])

    del tarefa, saida
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

del estado_arquivo

In [ ]:
from importlib.metadata import version

INFO = {
    'gerado_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'versoes': {p: version(p) for p in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy')},
    'python': platform.python_version(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'backbone': {
        'repositorio': REPO, 'arquivo': ARQUIVO, 'revisao': REVISAO, 'sha256': sha_local,
        'commits_com_o_arquivo': len(historico), 'hashes_distintos_no_historico': hashes_distintos,
    },
    'receita': 'config.yaml do commit 918b9f1 do checkpoint ajustado da IBM',
    'drive_montado': DRIVE_OK,
    'regimes': REGISTRO,
}
caminho = os.path.join(RES, 'ambiente_regimes.json')
json.dump(INFO, open(caminho, 'w'), indent=2, ensure_ascii=False)
print(json.dumps(INFO, indent=2, ensure_ascii=False))
print(f'\nGravado em {caminho}')
print('Cole aqui no Claude a saída desta célula e das células 3 e 4.')